In [ ]:
import numpy as np
from scipy.optimize import brentq
import pandas as pd
import os


def compute_irr(cashflows, lo=-0.5, hi=10.0, tol=1e-12):
    """Compute the Internal Rate of Return (IRR) of a series of cashflows.

    Uses scipy.optimize.brentq to find the rate r where NPV = 0.
    NPV = sum( cf[t] / (1+r)^t  for t in 0..N )
    """
    def npv_at_rate(r):
        return sum(cf / (1 + r) ** t for t, cf in enumerate(cashflows))

    return brentq(npv_at_rate, lo, hi, xtol=tol, maxiter=1000)


# Determine data path
if os.path.exists('/workspace/data/'):
    data_path = '/workspace/data/'
else:
    data_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'environment', 'data') + '/'
    if not os.path.exists(data_path):
        data_path = './data/'

print('Data path:', data_path)

In [ ]:
###############################################################################
# MACROHARD FINANCIAL MODEL - 10 Year Forecast
###############################################################################

# Years: 0 (Day 0) through 10
# Year indices 1-10 correspond to calendar years 2016-2025

years = list(range(0, 11))  # 0 to 10

# ============================================================================
# ASSUMPTIONS
# ============================================================================

# Day 0
initial_equity = 30_000_000
initial_debt = 20_000_000
initial_cash = 3_000_000
initial_ar = 3_000_000
initial_inventory = 4_000_000
initial_ppe = 40_000_000

# Revenue
year1_sales = 25_000_000
sales_growth_fixed = 3_000_000
sales_growth_pct = 0.08

# COGS
cogs_pct = 0.60

# Expenses
year1_expenses = 3_000_000
expense_growth_pct = 0.07

# CapEx
capex_year = 4
capex_amount = 10_000_000
capex_equity_pct = 0.90

# Depreciation
original_ppe_life = 10
new_ppe_life = 5
original_dep_annual = initial_ppe / original_ppe_life  # 4,000,000
new_dep_annual = capex_amount / new_ppe_life  # 2,000,000

# Debt
interest_rate = 0.065
principal_repayment = 2_500_000

# Working Capital
inventory_pct = 0.30  # of COGS
ar_pct = 0.10  # of Sales
ap_cogs_pct = 0.12  # of COGS
ap_exp_pct = 0.15  # of Expenses

# Distributions
min_cash_balance = 4_000_000

print('Assumptions loaded.')

In [ ]:
###############################################################################
# BUILD THE MODEL
###############################################################################

# Initialize arrays (index 0 = Day 0, indices 1-10 = Years 1-10)
sales = [0.0] * 11
cogs = [0.0] * 11
expenses = [0.0] * 11
depreciation = [0.0] * 11
interest = [0.0] * 11
net_profit = [0.0] * 11

# Balance Sheet items
cash = [0.0] * 11
accounts_receivable = [0.0] * 11
inventory = [0.0] * 11
ppe_gross = [0.0] * 11
accum_depreciation = [0.0] * 11
ppe_net = [0.0] * 11
total_assets = [0.0] * 11

accounts_payable = [0.0] * 11
debt_balance = [0.0] * 11
equity_balance = [0.0] * 11
retained_earnings = [0.0] * 11
total_liabilities_equity = [0.0] * 11

# Cash Flow items
distributions = [0.0] * 11
equity_injections = [0.0] * 11
capex_arr = [0.0] * 11
principal_repayments = [0.0] * 11

# Debt schedule
debt_opening = [0.0] * 11
debt_closing = [0.0] * 11

# ============================================================================
# DAY 0 BALANCE SHEET
# ============================================================================
cash[0] = initial_cash
accounts_receivable[0] = initial_ar
inventory[0] = initial_inventory
ppe_gross[0] = initial_ppe
accum_depreciation[0] = 0
ppe_net[0] = initial_ppe
total_assets[0] = cash[0] + accounts_receivable[0] + inventory[0] + ppe_net[0]

accounts_payable[0] = 0
debt_balance[0] = initial_debt
equity_balance[0] = initial_equity
retained_earnings[0] = 0
total_liabilities_equity[0] = accounts_payable[0] + debt_balance[0] + equity_balance[0] + retained_earnings[0]

# Equity injections on Day 0
equity_injections[0] = initial_equity

print(f'Day 0 Total Assets: ${total_assets[0]:,.0f}')
print(f'Day 0 Total L+E: ${total_liabilities_equity[0]:,.0f}')
print(f'Balance sheet balances: {abs(total_assets[0] - total_liabilities_equity[0]) < 0.01}')

In [ ]:
# ============================================================================
# REVENUE SCHEDULE
# ============================================================================
sales[1] = year1_sales
for y in range(2, 11):
    growth = min(sales_growth_fixed, sales_growth_pct * sales[y-1])
    sales[y] = sales[y-1] + growth

print('Sales schedule:')
for y in range(1, 11):
    print(f'  Year {y}: ${sales[y]:,.2f}')

# ============================================================================
# COGS SCHEDULE
# ============================================================================
for y in range(1, 11):
    cogs[y] = cogs_pct * sales[y]

print('\nCOGS schedule:')
for y in range(1, 11):
    print(f'  Year {y}: ${cogs[y]:,.2f}')

total_cogs = sum(cogs[1:11])
print(f'\nTotal COGS over 10 years: ${total_cogs:,.2f}')

# ============================================================================
# EXPENSES SCHEDULE
# ============================================================================
expenses[1] = year1_expenses
for y in range(2, 11):
    expenses[y] = expenses[y-1] * (1 + expense_growth_pct)

print('\nExpenses schedule:')
for y in range(1, 11):
    print(f'  Year {y}: ${expenses[y]:,.2f}')

In [ ]:
# ============================================================================
# DEPRECIATION SCHEDULE
# ============================================================================
# Original PPE: $40M over 10 years = $4M/year (Years 1-10)
# New PPE: $10M over 5 years = $2M/year (Years 5-9)
# New PPE is purchased at end of Year 4, so depreciation starts Year 5

for y in range(1, 11):
    dep = original_dep_annual  # $4M per year for original PPE
    if y >= 5 and y <= 9:  # New PPE depreciates years 5-9
        dep += new_dep_annual
    depreciation[y] = dep

print('Depreciation schedule:')
for y in range(1, 11):
    print(f'  Year {y}: ${depreciation[y]:,.2f}')

# ============================================================================
# DEBT SCHEDULE
# ============================================================================
debt_opening[0] = 0
debt_closing[0] = initial_debt

for y in range(1, 11):
    debt_opening[y] = debt_closing[y-1]
    # Interest on opening balance
    interest[y] = debt_opening[y] * interest_rate
    # Principal repayment (only if debt remains)
    principal_repayments[y] = min(principal_repayment, debt_opening[y])
    debt_closing[y] = debt_opening[y] - principal_repayments[y]

print('\nDebt schedule:')
for y in range(1, 11):
    print(f'  Year {y}: Opening=${debt_opening[y]:,.0f}, Interest=${interest[y]:,.0f}, Repayment=${principal_repayments[y]:,.0f}, Closing=${debt_closing[y]:,.0f}')

total_interest = sum(interest[1:11])
print(f'\nTotal Interest over 10 years: ${total_interest:,.2f}')

In [ ]:
# ============================================================================
# CAPEX AND EQUITY INJECTION
# ============================================================================
capex_arr[capex_year] = capex_amount
equity_injections[capex_year] = capex_amount * capex_equity_pct  # $9,000,000

print(f'Year {capex_year} CapEx: ${capex_arr[capex_year]:,.0f}')
print(f'Year {capex_year} Equity Injection: ${equity_injections[capex_year]:,.0f}')
print(f'Year {capex_year} Cash portion of CapEx: ${capex_amount - equity_injections[capex_year]:,.0f}')

In [ ]:
# ============================================================================
# PROFIT AND LOSS STATEMENT
# ============================================================================
gross_profit = [0.0] * 11
ebitda = [0.0] * 11
ebit = [0.0] * 11

for y in range(1, 11):
    gross_profit[y] = sales[y] - cogs[y]
    ebitda[y] = gross_profit[y] - expenses[y]
    ebit[y] = ebitda[y] - depreciation[y]
    net_profit[y] = ebit[y] - interest[y]  # No tax

print('P&L Summary:')
print(f'{"Year":>6} {"Sales":>15} {"COGS":>15} {"GP":>15} {"Expenses":>15} {"Deprec":>15} {"Interest":>15} {"Net Profit":>15}')
for y in range(1, 11):
    print(f'{y:>6} {sales[y]:>15,.0f} {cogs[y]:>15,.0f} {gross_profit[y]:>15,.0f} {expenses[y]:>15,.0f} {depreciation[y]:>15,.0f} {interest[y]:>15,.0f} {net_profit[y]:>15,.0f}')

In [ ]:
# ============================================================================
# WORKING CAPITAL
# ============================================================================
for y in range(1, 11):
    accounts_receivable[y] = ar_pct * sales[y]
    inventory[y] = inventory_pct * cogs[y]
    accounts_payable[y] = ap_cogs_pct * cogs[y] + ap_exp_pct * expenses[y]

# Working capital changes
wc_change = [0.0] * 11  # Increase in WC is a cash outflow (negative for cash)
for y in range(1, 11):
    delta_ar = accounts_receivable[y] - accounts_receivable[y-1]
    delta_inv = inventory[y] - inventory[y-1]
    delta_ap = accounts_payable[y] - accounts_payable[y-1]
    # Increase in AR/Inv = cash outflow, Increase in AP = cash inflow
    wc_change[y] = -delta_ar - delta_inv + delta_ap

print('Working Capital:')
for y in range(1, 11):
    print(f'  Year {y}: AR=${accounts_receivable[y]:,.0f}, Inv=${inventory[y]:,.0f}, AP=${accounts_payable[y]:,.0f}, WC Change=${wc_change[y]:,.0f}')

In [ ]:
# ============================================================================
# PPE SCHEDULE
# ============================================================================
for y in range(1, 11):
    ppe_gross[y] = ppe_gross[y-1] + capex_arr[y]
    accum_depreciation[y] = accum_depreciation[y-1] + depreciation[y]
    ppe_net[y] = ppe_gross[y] - accum_depreciation[y]

print('PPE Schedule:')
for y in range(0, 11):
    print(f'  Year {y}: Gross=${ppe_gross[y]:,.0f}, Accum Dep=${accum_depreciation[y]:,.0f}, Net=${ppe_net[y]:,.0f}')

In [ ]:
# ============================================================================
# CASH FLOW STATEMENT AND DISTRIBUTIONS
# ============================================================================

net_cash_flow_before_dist = [0.0] * 11

for y in range(1, 11):
    # Net cash flow before distributions =
    #   Net Profit + Depreciation (add back non-cash) + WC changes
    #   - CapEx + Equity Injection - Principal Repayment
    net_cf = net_profit[y] + depreciation[y] + wc_change[y] - capex_arr[y] + equity_injections[y] - principal_repayments[y]
    net_cash_flow_before_dist[y] = net_cf

    # Cash before distributions
    cash_before_dist = cash[y-1] + net_cf

    # Distributions: anything above $4M
    if cash_before_dist > min_cash_balance:
        distributions[y] = cash_before_dist - min_cash_balance
    else:
        distributions[y] = 0

    cash[y] = cash_before_dist - distributions[y]

print('Cash Flow Summary:')
print(f'{"Year":>6} {"Opening Cash":>15} {"Net CF":>15} {"Cash Pre-Dist":>15} {"Distribution":>15} {"Closing Cash":>15}')
for y in range(1, 11):
    opening = cash[y-1]
    cash_pre = opening + net_cash_flow_before_dist[y]
    print(f'{y:>6} {opening:>15,.0f} {net_cash_flow_before_dist[y]:>15,.0f} {cash_pre:>15,.0f} {distributions[y]:>15,.0f} {cash[y]:>15,.0f}')

total_distributions = sum(distributions[1:11])
print(f'\nTotal Distributions over 10 years: ${total_distributions:,.2f}')

In [ ]:
# ============================================================================
# BALANCE SHEET
# ============================================================================
# Equity balance and retained earnings
for y in range(1, 11):
    equity_balance[y] = equity_balance[y-1] + equity_injections[y]
    # Retained earnings = previous RE + net profit - distributions
    retained_earnings[y] = retained_earnings[y-1] + net_profit[y] - distributions[y]

    debt_balance[y] = debt_closing[y]

    total_assets[y] = cash[y] + accounts_receivable[y] + inventory[y] + ppe_net[y]
    total_liabilities_equity[y] = accounts_payable[y] + debt_balance[y] + equity_balance[y] + retained_earnings[y]

print('Balance Sheet Summary:')
print(f'{"Year":>6} {"Cash":>12} {"AR":>12} {"Inv":>12} {"PPE Net":>12} {"Total A":>12} | {"AP":>12} {"Debt":>12} {"Equity":>12} {"RE":>12} {"Total L+E":>12} {"Bal?":>6}')
for y in range(0, 11):
    bal = abs(total_assets[y] - total_liabilities_equity[y]) < 0.01
    total_equity = equity_balance[y] + retained_earnings[y]
    print(f'{y:>6} {cash[y]:>12,.0f} {accounts_receivable[y]:>12,.0f} {inventory[y]:>12,.0f} {ppe_net[y]:>12,.0f} {total_assets[y]:>12,.0f} | {accounts_payable[y]:>12,.0f} {debt_balance[y]:>12,.0f} {equity_balance[y]:>12,.0f} {retained_earnings[y]:>12,.0f} {total_liabilities_equity[y]:>12,.0f} {str(bal):>6}')

print('\nTotal Equity (Shareholders Funds) at end of Year 10:')
total_equity_y10 = equity_balance[10] + retained_earnings[10]
print(f'  Equity Balance: ${equity_balance[10]:,.2f}')
print(f'  Retained Earnings: ${retained_earnings[10]:,.2f}')
print(f'  Total: ${total_equity_y10:,.2f}')

In [ ]:
# ============================================================================
# IRR CALCULATION
# ============================================================================
# Equity cash flows: negative for injections, positive for distributions
# Year 0: -$30,000,000 (initial equity investment)
# Year 4: -$9,000,000 (additional equity injection for CapEx)
# Years 1-10: distributions (positive)

equity_cf = [0.0] * 11
equity_cf[0] = -initial_equity  # Day 0 investment
for y in range(1, 11):
    equity_cf[y] = distributions[y] - equity_injections[y]

print('Equity Cash Flows:')
for y in range(0, 11):
    print(f'  Year {y}: ${equity_cf[y]:,.2f}')

# IRR using scipy.optimize.brentq (replaces numpy_financial)
irr = compute_irr(equity_cf)

print(f'\nEquity IRR: {irr*100:.4f}%')
print(f'Equity IRR (to nearest 0.01%): {irr*100:.2f}%')

In [ ]:
# ============================================================================
# QUESTION 15: Find growth rate for 13% IRR
# ============================================================================
# Rebuild the model with a variable sales growth percentage (not min of fixed/pct)

def build_model_pct_growth(growth_rate):
    """Build the full model with a fixed percentage sales growth rate."""
    s = [0.0] * 11
    s[1] = year1_sales
    for y in range(2, 11):
        s[y] = s[y-1] * (1 + growth_rate)

    c = [cogs_pct * s[y] for y in range(11)]

    e = [0.0] * 11
    e[1] = year1_expenses
    for y in range(2, 11):
        e[y] = e[y-1] * (1 + expense_growth_pct)

    dep = [0.0] * 11
    for y in range(1, 11):
        dep[y] = original_dep_annual
        if 5 <= y <= 9:
            dep[y] += new_dep_annual

    # Debt
    d_closing = [0.0] * 11
    d_closing[0] = initial_debt
    intr = [0.0] * 11
    pr = [0.0] * 11
    for y in range(1, 11):
        d_opening = d_closing[y-1]
        intr[y] = d_opening * interest_rate
        pr[y] = min(principal_repayment, d_opening)
        d_closing[y] = d_opening - pr[y]

    # Net Profit
    np_arr = [0.0] * 11
    for y in range(1, 11):
        gp = s[y] - c[y]
        ebitda_val = gp - e[y]
        ebit_val = ebitda_val - dep[y]
        np_arr[y] = ebit_val - intr[y]

    # Working Capital
    ar = [0.0] * 11
    inv = [0.0] * 11
    ap = [0.0] * 11
    ar[0] = initial_ar
    inv[0] = initial_inventory
    ap[0] = 0
    for y in range(1, 11):
        ar[y] = ar_pct * s[y]
        inv[y] = inventory_pct * c[y]
        ap[y] = ap_cogs_pct * c[y] + ap_exp_pct * e[y]

    wc = [0.0] * 11
    for y in range(1, 11):
        wc[y] = -(ar[y] - ar[y-1]) - (inv[y] - inv[y-1]) + (ap[y] - ap[y-1])

    # CapEx and equity injections
    cx = [0.0] * 11
    ei = [0.0] * 11
    cx[capex_year] = capex_amount
    ei[capex_year] = capex_amount * capex_equity_pct

    # Cash and distributions
    ca = [0.0] * 11
    ca[0] = initial_cash
    dist = [0.0] * 11

    for y in range(1, 11):
        net_cf = np_arr[y] + dep[y] + wc[y] - cx[y] + ei[y] - pr[y]
        cash_pre = ca[y-1] + net_cf
        if cash_pre > min_cash_balance:
            dist[y] = cash_pre - min_cash_balance
        ca[y] = cash_pre - dist[y]

    # Equity cash flows
    ecf = [0.0] * 11
    ecf[0] = -initial_equity
    for y in range(1, 11):
        ecf[y] = dist[y] - ei[y]

    return ecf

# Binary search for growth rate that gives 13% IRR
target_irr = 0.13
low, high = 0.01, 0.20

for _ in range(200):
    mid = (low + high) / 2
    ecf = build_model_pct_growth(mid)
    irr_val = compute_irr(ecf)
    if irr_val < target_irr:
        low = mid
    else:
        high = mid

q15_growth_rate = (low + high) / 2
print(f'Growth rate for 13% IRR: {q15_growth_rate*100:.4f}%')
print(f'Growth rate (to nearest 0.01%): {q15_growth_rate*100:.2f}%')

# Verify
ecf_verify = build_model_pct_growth(q15_growth_rate)
irr_verify = compute_irr(ecf_verify)
print(f'Verified IRR: {irr_verify*100:.4f}%')

In [ ]:
# ============================================================================
# ANSWER ALL QUESTIONS
# ============================================================================

def match_answer(value, options):
    """Match a computed value to the closest option."""
    # Round to nearest thousand
    rounded = round(value / 1000) * 1000
    best_letter = None
    best_diff = float('inf')
    for letter, opt_val in options.items():
        diff = abs(rounded - opt_val)
        if diff < best_diff:
            best_diff = diff
            best_letter = letter
    return best_letter

def match_pct_answer(value_pct, options):
    """Match a percentage value to the closest option."""
    best_letter = None
    best_diff = float('inf')
    for letter, opt_val in options.items():
        diff = abs(value_pct - opt_val)
        if diff < best_diff:
            best_diff = diff
            best_letter = letter
    return best_letter

# Q1: Sales in Year 3 (to nearest $1000)
q1_val = sales[3]
q1_options = {'A': 29159000, 'B': 29160000, 'C': 29161000, 'D': 29162000, 'E': 29163000, 'F': 29164000}
q1 = match_answer(q1_val, q1_options)
print(f'Q1: Sales Year 3 = ${q1_val:,.2f} -> rounded to ${round(q1_val/1000)*1000:,.0f} -> {q1}')

# Q2: Sales in Year 9 (to nearest $1000)
q2_val = sales[9]
q2_options = {'A': 45669000, 'B': 45670000, 'C': 45671000, 'D': 45672000, 'E': 45673000, 'F': 45674000}
q2 = match_answer(q2_val, q2_options)
print(f'Q2: Sales Year 9 = ${q2_val:,.2f} -> rounded to ${round(q2_val/1000)*1000:,.0f} -> {q2}')

# Q3: Total COGS over 10 years (to nearest $1000)
q3_val = total_cogs
q3_options = {'A': 216046000, 'B': 216047000, 'C': 216048000, 'D': 216049000, 'E': 216050000, 'F': 216051000}
q3 = match_answer(q3_val, q3_options)
print(f'Q3: Total COGS = ${q3_val:,.2f} -> rounded to ${round(q3_val/1000)*1000:,.0f} -> {q3}')

# Q4: Year 7 Expenses as % of Year 7 Sales (to nearest 0.01%)
q4_val = (expenses[7] / sales[7]) * 100
q4_options = {'A': 11.30, 'B': 11.31, 'C': 11.32, 'D': 11.33, 'E': 11.34, 'F': 11.35}
q4 = match_pct_answer(q4_val, q4_options)
print(f'Q4: Year 7 Exp/Sales = {q4_val:.4f}% -> {q4}')

# Q5: Total Interest over 10 years (to nearest $1000)
q5_val = total_interest
q5_options = {'A': 5847000, 'B': 5848000, 'C': 5849000, 'D': 5850000, 'E': 5851000, 'F': 5852000}
q5 = match_answer(q5_val, q5_options)
print(f'Q5: Total Interest = ${q5_val:,.2f} -> rounded to ${round(q5_val/1000)*1000:,.0f} -> {q5}')

# Q6: Book value of all PPE at end of Year 7 (to nearest $million)
q6_val = ppe_net[7]
q6_options = {'A': 15000000, 'B': 16000000, 'C': 17000000, 'D': 18000000, 'E': 19000000, 'F': 20000000}
q6_nearest_m = round(q6_val / 1000000) * 1000000
q6 = match_answer(q6_val, q6_options)
print(f'Q6: PPE Net Year 7 = ${q6_val:,.2f} -> nearest $M = ${q6_nearest_m:,.0f} -> {q6}')

# Q7: Inventory at end of Year 5 (to nearest $1000)
q7_val = inventory[5]
q7_options = {'A': 6117000, 'B': 6118000, 'C': 6119000, 'D': 6120000, 'E': 6121000, 'F': 6122000}
q7 = match_answer(q7_val, q7_options)
print(f'Q7: Inventory Year 5 = ${q7_val:,.2f} -> rounded to ${round(q7_val/1000)*1000:,.0f} -> {q7}')

# Q8: Total Assets at end of Year 6 (to nearest $1000)
q8_val = total_assets[6]
q8_options = {'A': 36285000, 'B': 36286000, 'C': 36287000, 'D': 36288000, 'E': 36289000, 'F': 36290000}
q8 = match_answer(q8_val, q8_options)
print(f'Q8: Total Assets Year 6 = ${q8_val:,.2f} -> rounded to ${round(q8_val/1000)*1000:,.0f} -> {q8}')

# Q9: Accounts Payable at end of Year 4 (to nearest $1000)
q9_val = accounts_payable[4]
q9_options = {'A': 2817000, 'B': 2818000, 'C': 2819000, 'D': 2820000, 'E': 2821000, 'F': 2822000}
q9 = match_answer(q9_val, q9_options)
print(f'Q9: AP Year 4 = ${q9_val:,.2f} -> rounded to ${round(q9_val/1000)*1000:,.0f} -> {q9}')

# Q10: Net Profit (before distributions) in Year 9 (to nearest $1000)
q10_val = net_profit[9]
q10_options = {'A': 7111000, 'B': 7112000, 'C': 7113000, 'D': 7114000, 'E': 7115000, 'F': 7116000}
q10 = match_answer(q10_val, q10_options)
print(f'Q10: Net Profit Year 9 = ${q10_val:,.2f} -> rounded to ${round(q10_val/1000)*1000:,.0f} -> {q10}')

# Q11: Net Cash Flow (before distributions) in Year 4 (to nearest $1000)
q11_val = net_cash_flow_before_dist[4]
q11_options = {'A': 4158000, 'B': 4159000, 'C': 4160000, 'D': 4161000, 'E': 4162000, 'F': 4163000}
q11 = match_answer(q11_val, q11_options)
print(f'Q11: Net CF before dist Year 4 = ${q11_val:,.2f} -> rounded to ${round(q11_val/1000)*1000:,.0f} -> {q11}')

# Q12: Total distributions to equity over 10 years (to nearest $1000)
q12_val = total_distributions
q12_options = {'A': 72437000, 'B': 72438000, 'C': 72439000, 'D': 72440000, 'E': 72441000, 'F': 72442000}
q12 = match_answer(q12_val, q12_options)
print(f'Q12: Total Distributions = ${q12_val:,.2f} -> rounded to ${round(q12_val/1000)*1000:,.0f} -> {q12}')

# Q13: Total Equity (Shareholders' Funds) at end of Year 10 (to nearest $1000)
q13_val = total_equity_y10
q13_options = {'A': 13295000, 'B': 13296000, 'C': 13297000, 'D': 13298000, 'E': 13299000, 'F': 13300000}
q13 = match_answer(q13_val, q13_options)
print(f'Q13: Total Equity Year 10 = ${q13_val:,.2f} -> rounded to ${round(q13_val/1000)*1000:,.0f} -> {q13}')

# Q14: IRR of equity cash flows (to nearest 0.01%)
q14_val = irr * 100
q14_options = {'A': 11.88, 'B': 11.89, 'C': 11.90, 'D': 11.91, 'E': 11.92, 'F': 11.93}
q14 = match_pct_answer(q14_val, q14_options)
print(f'Q14: IRR = {q14_val:.4f}% -> {q14}')

# Q15: Growth rate for 13% IRR (to nearest 0.01%)
q15_val = q15_growth_rate * 100
q15_options = {'A': 8.59, 'B': 8.60, 'C': 8.61, 'D': 8.62, 'E': 8.63, 'F': 8.64}
q15 = match_pct_answer(q15_val, q15_options)
print(f'Q15: Growth rate = {q15_val:.4f}% -> {q15}')

In [ ]:
# ============================================================================
# SET FINAL ANSWERS
# ============================================================================
answers = {
    "question1": q1,
    "question2": q2,
    "question3": q3,
    "question4": q4,
    "question5": q5,
    "question6": q6,
    "question7": q7,
    "question8": q8,
    "question9": q9,
    "question10": q10,
    "question11": q11,
    "question12": q12,
    "question13": q13,
    "question14": q14,
    "question15": q15,
}

print('\nFinal Answers:')
for k, v in answers.items():
    print(f'  {k}: {v}')